# Klasifikasi Lahan Sawah dan Non-Sawah dengan Random Forest

Notebook ini menggunakan 10 fitur Sentinel-2A: B2, B3, B4, B8, B11, B12, NDVI, NDWI, MNDWI, dan NDBI. Satu data berarti satu piksel berlabel. GeoJSON berisi 50 petak sawah dan 50 petak non-sawah.

In [ ]:
!pip -q install earthengine-api requests
import ee
import requests

# Ganti dengan Google Cloud Project ID yang sudah mengaktifkan Earth Engine.
PROJECT_GEE = 'ID-PROJECT-GEE'
START, END = '2024-06-01', '2024-09-30'
MAX_AWAN = 40
HANYA_S2A = True
OUT_TIF = 'sentinel2A_komposit.tif'

# Jalankan autentikasi sekali pada environment yang belum terhubung ke Earth Engine.
ee.Authenticate()
ee.Initialize(project=PROJECT_GEE)

AOI = ee.Geometry.Rectangle([112.7060, -7.1495, 112.7270, -7.1325])

koleksi = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(AOI)
           .filterDate(START, END)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_AWAN)))
print('Semua Sentinel-2 L2A:', koleksi.size().getInfo(), 'citra')
if HANYA_S2A:
    koleksi = koleksi.filter(ee.Filter.eq('SPACECRAFT_NAME', 'Sentinel-2A'))
n = koleksi.size().getInfo()
print('Dipakai             :', n, 'citra')
if n == 0:
    raise RuntimeError('Tidak ada citra. Ubah periode, MAX_AWAN, atau HANYA_S2A.')

def mask_awan(img):
    scl = img.select('SCL')
    bagus = (scl.neq(0).And(scl.neq(1)).And(scl.neq(3)).And(scl.neq(8))
             .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    return img.updateMask(bagus)

BAND = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
komposit = koleksi.map(mask_awan).select(BAND).median().divide(10000)
ndvi = komposit.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = komposit.normalizedDifference(['B3', 'B8']).rename('NDWI')
mndwi = komposit.normalizedDifference(['B3', 'B11']).rename('MNDWI')
ndbi = komposit.normalizedDifference(['B11', 'B8']).rename('NDBI')
citra = komposit.addBands([ndvi, ndwi, mndwi, ndbi]).toFloat().clip(AOI)

url = citra.getDownloadURL({
    'region': AOI,
    'scale': 10,
    'crs': 'EPSG:32749',
    'format': 'GEO_TIFF',
})
r = requests.get(url, timeout=900)
r.raise_for_status()
if r.content[:2] not in (b'II', b'MM'):
    raise RuntimeError(r.text[:500])
with open(OUT_TIF, 'wb') as file:
    file.write(r.content)
print(f'Tersimpan {OUT_TIF}: {len(r.content) / 1e6:.1f} MB')
print('Urutan fitur:', BAND + ['NDVI', 'NDWI', 'MNDWI', 'NDBI'])

try:
    from importlib import import_module
    files = import_module('google.colab.files')
    files.download(OUT_TIF)
except ImportError:
    print(f'File tersedia di folder kerja: {OUT_TIF}')


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\laragon\bin\python\python-3.10\python.exe -m pip install --upgrade pip
c:\Users\ARIEL\AppData\Local\Programs\Python\Python39\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
c:\Users\ARIEL\AppData\Local\Programs\Python\Python39\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
c:\Us

In [ ]:
!pip -q install rasterio shapely scikit-learn matplotlib
import json
from pathlib import Path

import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.warp import transform
from shapely.geometry import Polygon, mapping
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score

TIF = Path('sentinel2A_komposit.tif')
OUT_KLAS = Path('klasifikasi_sawah.tif')
OUT_GAMBAR = Path('segmentasi_sawah_non_sawah.png')
FILE_SAWAH = Path('sawah.geojson')
FILE_NON_SAWAH = Path('nonsawah.geojson')
NAMA_BAND = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'MNDWI', 'NDBI']
BUFFER_M = 5


def baca_geojson(path):
    if not path.exists():
        raise FileNotFoundError(f'File tidak ditemukan: {path.resolve()}')
    with path.open(encoding='utf-8') as file:
        data = json.load(file)
    if data.get('type') != 'FeatureCollection':
        raise ValueError(f'{path} harus bertipe FeatureCollection')
    if not data.get('features'):
        raise ValueError(f'{path} tidak berisi feature')
    return data


TITIK_SAWAH = baca_geojson(FILE_SAWAH)
TITIK_NON_SAWAH = baca_geojson(FILE_NON_SAWAH)
print(f'Petak sawah dari GeoJSON     : {len(TITIK_SAWAH["features"])}')
print(f'Petak non-sawah dari GeoJSON : {len(TITIK_NON_SAWAH["features"])}')
print(f'Total petak GeoJSON          : {len(TITIK_SAWAH["features"]) + len(TITIK_NON_SAWAH["features"])}')

with rasterio.open(TIF) as src:
    arr = src.read().astype('float32')
    profil = src.profile
    tf, crs = src.transform, src.crs

n_band, H, W = arr.shape
if n_band != len(NAMA_BAND):
    raise ValueError(f'TIF memiliki {n_band} band, tetapi model memerlukan {len(NAMA_BAND)} fitur')
valid = np.isfinite(arr).all(axis=0) & (arr[:6].sum(axis=0) > 0)
print(f'Ukuran citra                : {W} x {H} piksel')
print(f'Jumlah band/fitur           : {n_band}')
print(f'CRS                         : {crs}')


def petak_ke_polygon(fc):
    hasil = []
    for feature in fc.get('features', []):
        geometry = feature.get('geometry') or {}
        if geometry.get('type') == 'LineString':
            ring = geometry['coordinates']
        elif geometry.get('type') == 'Polygon':
            ring = geometry['coordinates'][0]
        else:
            continue
        if len(ring) < 3:
            continue
        lon = [point[0] for point in ring]
        lat = [point[1] for point in ring]
        xs, ys = transform('EPSG:4326', crs, lon, lat)
        polygon = Polygon(zip(xs, ys))
        if not polygon.is_valid:
            polygon = polygon.buffer(0)
        if polygon.is_empty or polygon.area == 0:
            continue
        kecil = polygon.buffer(-BUFFER_M)
        hasil.append(kecil if not kecil.is_empty and kecil.area > 0 else polygon)
    return hasil


def ambil(fc, kelas, offset):
    polygons = petak_ke_polygon(fc)
    if not polygons:
        return np.empty((0, n_band), dtype='float32'), np.empty(0, dtype=int), np.empty(0, dtype=int)
    shapes = [(mapping(polygon), offset + index + 1) for index, polygon in enumerate(polygons)]
    ids = rasterize(shapes, out_shape=(H, W), transform=tf, fill=0, dtype='int32')
    mask = (ids > 0) & valid
    features = arr[:, mask].T
    groups = ids[mask]
    print(f'Kelas {kelas}: {len(polygons)} petak, {len(np.unique(groups))} petak masuk, {len(features)} piksel')
    return features, np.full(len(features), kelas, dtype=int), groups


X_sawah, y_sawah, g_sawah = ambil(TITIK_SAWAH, 1, 0)
X_non_sawah, y_non_sawah, g_non_sawah = ambil(TITIK_NON_SAWAH, 0, 10000)
X = np.vstack([X_sawah, X_non_sawah])
y = np.concatenate([y_sawah, y_non_sawah])
grup = np.concatenate([g_sawah, g_non_sawah])

if len(np.unique(y)) != 2:
    raise ValueError('Kedua kelas harus memiliki piksel valid di dalam citra')
if len(np.unique(grup)) < 4:
    raise ValueError('Minimal empat petak harus tersedia untuk split train/testing')

jumlah_fitur = X.shape[1]
jumlah_data = X.shape[0]
jumlah_piksel_sawah = int((y == 1).sum())
jumlah_piksel_non_sawah = int((y == 0).sum())
jumlah_petak = len(np.unique(grup))
print('\n=== RINGKASAN DATA ===')
print(f'Jumlah data piksel berlabel  : {jumlah_data}')
print(f'Jumlah fitur per piksel      : {jumlah_fitur}')
print(f'Ukuran X                    : {X.shape}')
print(f'Piksel label sawah (1)       : {jumlah_piksel_sawah}')
print(f'Piksel label non-sawah (0)   : {jumlah_piksel_non_sawah}')
print(f'Jumlah petak yang terbaca    : {jumlah_petak}')
print(f'Nama fitur                   : {NAMA_BAND}')

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_index, test_index = next(splitter.split(X, y, grup))
X_train, X_test = X[train_index], X[test_index]
y_train, y_test = y[train_index], y[test_index]
petak_train = len(np.unique(grup[train_index]))
petak_test = len(np.unique(grup[test_index]))

print('\n=== UKURAN TRAINING DAN TESTING ===')
print(f'X_train shape                : {X_train.shape}')
print(f'X_test shape                 : {X_test.shape}')
print(f'y_train shape                : {y_train.shape}')
print(f'y_test shape                 : {y_test.shape}')
print(f'Data train                   : {len(X_train)} piksel dari {petak_train} petak')
print(f'Data testing                 : {len(X_test)} piksel dari {petak_test} petak')
print(f'Train sawah/non-sawah        : {(y_train == 1).sum()} / {(y_train == 0).sum()}')
print(f'Test sawah/non-sawah         : {(y_test == 1).sum()} / {(y_test == 0).sum()}')

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
akurasi = accuracy_score(y_test, pred)
kappa = cohen_kappa_score(y_test, pred)
matriks = confusion_matrix(y_test, pred, labels=[0, 1])

print('\n=== HASIL RANDOM FOREST ===')
print(f'Akurasi                     : {akurasi:.4f}')
print(f'Kappa                       : {kappa:.4f}')
print('Matriks kebingungan (baris aktual, kolom prediksi):')
print(matriks)
print('\nKepentingan fitur:')
peringkat_fitur = sorted(zip(NAMA_BAND, rf.feature_importances_), key=lambda item: -item[1])
for nama_fitur, nilai in peringkat_fitur:
    print(f'  {nama_fitur:8s} {nilai:.4f} ({nilai * 100:.2f}%)')

# Latih ulang model menggunakan seluruh data berlabel.
rf.fit(X, y)
flat = arr.reshape(n_band, -1)
hasil_flat = np.full(H * W, 255, dtype='uint8')
valid_index = np.flatnonzero(valid.ravel())
for start in range(0, len(valid_index), 200000):
    index = valid_index[start:start + 200000]
    hasil_flat[index] = rf.predict(flat[:, index].T).astype('uint8')
hasil = hasil_flat.reshape(H, W)

for key in ('blockxsize', 'blockysize', 'tiled', 'interleave'):
    profil.pop(key, None)
profil.update(count=1, dtype='uint8', nodata=255, compress='lzw')
with rasterio.open(OUT_KLAS, 'w', **profil) as dst:
    dst.write(hasil, 1)

luas_px = abs(tf.a * tf.e)
n_sawah = int((hasil == 1).sum())
n_non = int((hasil == 0).sum())
luas_sawah_ha = n_sawah * luas_px / 10000
luas_non_ha = n_non * luas_px / 10000
persen_sawah = 100 * n_sawah / (n_sawah + n_non)
persen_non = 100 * n_non / (n_sawah + n_non)
print(f'\nHasil piksel klasifikasi sawah     : {n_sawah} (~{luas_sawah_ha:.2f} ha)')
print(f'Hasil piksel klasifikasi non-sawah : {n_non} (~{luas_non_ha:.2f} ha)')
print(f'Persentase sawah/non-sawah         : {persen_sawah:.2f}% / {persen_non:.2f}%')
print(f'Tersimpan                        : {OUT_KLAS.resolve()}')

In [ ]:
hasil_evaluasi = {
    'Jumlah fitur': jumlah_fitur,
    'Jumlah data piksel berlabel': jumlah_data,
    'Piksel label sawah': jumlah_piksel_sawah,
    'Piksel label non-sawah': jumlah_piksel_non_sawah,
    'Ukuran X': X.shape,
    'Ukuran X_train': X_train.shape,
    'Ukuran X_test': X_test.shape,
    'Data train': f'{len(X_train)} piksel ({petak_train} petak)',
    'Data testing': f'{len(X_test)} piksel ({petak_test} petak)',
    'Akurasi petak uji': f'{akurasi:.4f}',
    'Kappa': f'{kappa:.4f}',
    'Luas sawah (ha)': f'{luas_sawah_ha:.2f}',
    'Luas non-sawah (ha)': f'{luas_non_ha:.2f}',
}

print('| Metrik | Nilai |')
print('| :--- | ---: |')
for metrik, nilai in hasil_evaluasi.items():
    print(f'| {metrik} | {nilai} |')

print('\n| Kelas hasil klasifikasi | Piksel | Luas (ha) | Persentase |')
print('| :--- | ---: | ---: | ---: |')
print(f'| Sawah | {n_sawah:,} | {luas_sawah_ha:.2f} | {persen_sawah:.2f}% |')
print(f'| Non-sawah | {n_non:,} | {luas_non_ha:.2f} | {persen_non:.2f}% |')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

rgb = np.clip(np.nan_to_num(arr[[2, 1, 0]]).transpose(1, 2, 0) / 0.3, 0, 1)
kelas_cmap = ListedColormap(['#d95f02', '#1b9e77', '#ffffff'])
kelas_norm = BoundaryNorm([-0.5, 0.5, 1.5, 255.5], kelas_cmap.N)
kelas = np.ma.masked_where(hasil == 255, hasil)

fig, ax = plt.subplots(1, 3, figsize=(16, 5.5), constrained_layout=True)
ax[0].imshow(rgb)
ax[0].set_title('Citra Sentinel-2A (RGB)')
ax[1].imshow(kelas, cmap=kelas_cmap, norm=kelas_norm, interpolation='nearest')
ax[1].set_title('Segmentasi Sawah dan Non-Sawah')
ax[2].imshow(rgb)
ax[2].imshow(kelas, cmap=kelas_cmap, norm=kelas_norm, alpha=0.48, interpolation='nearest')
ax[2].set_title('Overlay Citra dan Hasil Segmentasi')
for axis in ax:
    axis.axis('off')
fig.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color='#1b9e77', label='Sawah'),
        plt.Rectangle((0, 0), 1, 1, color='#d95f02', label='Non-sawah'),
    ],
    loc='lower center', ncol=2, frameon=False,
)
fig.savefig(OUT_GAMBAR, dpi=200, bbox_inches='tight')
print(f'Gambar tersimpan: {OUT_GAMBAR.resolve()}')
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred,
    labels=[0, 1],
    display_labels=['Non-Sawah', 'Sawah'],
    cmap='Greens',
    ax=ax[0],
    colorbar=False,
)
ax[0].set_title('Confusion Matrix (petak uji)')

urut = np.argsort(rf.feature_importances_)
ax[1].barh(range(len(urut)), rf.feature_importances_[urut], color='#2ecc71', edgecolor='black')
ax[1].set_yticks(range(len(urut)))
ax[1].set_yticklabels([NAMA_BAND[index] for index in urut])
ax[1].set_xlabel('Nilai kepentingan')
ax[1].set_title('Kepentingan band dan indeks')
plt.tight_layout()
plt.show()